In [1]:
import os
import json
from getpass import getpass

import requests
import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
def jprint(value):
    print(json.dumps(value, indent=2, default=str))


def load_tiny_jeopardy():
    url = "https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/data/jeopardy_tiny.json"
    return requests.get(url, timeout=30).json()


def load_jeopardy_1k():
    url = "https://raw.githubusercontent.com/weaviate-tutorials/intro-workshop/main/data/jeopardy_1k.json"
    return requests.get(url, timeout=30).json()

In [4]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [5]:
COLLECTION_NAME = "QuestionHybrid"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

questions_hybrid = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small"
    ),
    properties=[
        wvc.config.Property(
            name="question",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="answer",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="category",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: QuestionHybrid


In [6]:
data = load_tiny_jeopardy()

print("Number of records:", len(data))
print(data[0])

Number of records: 10
{'Category': 'SCIENCE', 'Question': 'This organ removes excess glucose from the blood & stores it as glycogen', 'Answer': 'Liver'}


In [7]:
objects = []

for item in data:
    objects.append(
        {
            "question": item["Question"],
            "answer": item["Answer"],
            "category": item["Category"],
        }
    )

result = questions_hybrid.data.insert_many(objects)

if result.errors:
    print("Import errors:")
    jprint(result.errors)
else:
    print("Import completed.")

Import completed.


In [8]:
response = questions_hybrid.query.fetch_objects(
    limit=5,
    return_properties=["question", "answer", "category"],
)

for obj in response.objects:
    print(obj.uuid)
    print(obj.properties)
    print("-" * 80)

471d7d3f-b4e5-409a-a3eb-aee150df1dd5
{'answer': 'the diamondback rattler', 'question': 'Heaviest of all poisonous snakes is this North American rattlesnake', 'category': 'ANIMALS'}
--------------------------------------------------------------------------------
5d421084-f24f-4517-8c8f-c0027e748f54
{'answer': 'DNA', 'question': 'In 1953 Watson & Crick built a model of the molecular structure of this, the gene-carrying substance', 'category': 'SCIENCE'}
--------------------------------------------------------------------------------
956fe622-dd78-4fa4-bebe-5bcf73fedb9e
{'answer': 'Elephant', 'question': "It's the only living mammal in the order Proboseidea", 'category': 'ANIMALS'}
--------------------------------------------------------------------------------
a3b2a0be-1f57-40e9-bc84-f141ef464645
{'answer': 'Liver', 'question': 'This organ removes excess glucose from the blood & stores it as glycogen', 'category': 'SCIENCE'}
---------------------------------------------------------------

## Vector search -- semantic search

In [9]:
print("Vector search")

response = questions_hybrid.query.near_text(
    query="animal",
    limit=3,
    return_properties=["question", "answer", "category"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print(obj.properties)
    print("-" * 80)

Vector search
Distance: 0.6373440027236938
{'answer': 'Elephant', 'category': 'ANIMALS', 'question': "It's the only living mammal in the order Proboseidea"}
--------------------------------------------------------------------------------
Distance: 0.6576249599456787
{'answer': 'the nose or snout', 'question': 'The gavial looks very much like a crocodile except for this bodily feature', 'category': 'ANIMALS'}
--------------------------------------------------------------------------------
Distance: 0.6803133487701416
{'answer': 'Antelope', 'question': 'Weighing around a ton, the eland is the largest species of this animal in Africa', 'category': 'ANIMALS'}
--------------------------------------------------------------------------------


## MB25 -- keyword search

In [12]:
print("BM25 keyword search")

response = questions_hybrid.query.bm25(
    query="ANIMALS",
    query_properties=["category"],
    limit=3,
    return_properties=["question", "answer", "category"],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print(obj.properties)
    print("-" * 80)

BM25 keyword search
Score: 0.40628084540367126
{'answer': 'Elephant', 'question': "It's the only living mammal in the order Proboseidea", 'category': 'ANIMALS'}
--------------------------------------------------------------------------------
Score: 0.40628084540367126
{'answer': 'the nose or snout', 'question': 'The gavial looks very much like a crocodile except for this bodily feature', 'category': 'ANIMALS'}
--------------------------------------------------------------------------------
Score: 0.40628084540367126
{'answer': 'Antelope', 'question': 'Weighing around a ton, the eland is the largest species of this animal in Africa', 'category': 'ANIMALS'}
--------------------------------------------------------------------------------


## Hybrid search

In [13]:
print("Hybrid search alpha=0.5")

response = questions_hybrid.query.hybrid(
    query="animal",
    alpha=0.5,
    limit=3,
    return_properties=["question", "answer", "category"],
    return_metadata=MetadataQuery(score=True, explain_score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print(obj.properties)
    print("-" * 80)

Hybrid search alpha=0.5
Score: 0.9188180565834045
{'answer': 'Antelope', 'question': 'Weighing around a ton, the eland is the largest species of this animal in Africa', 'category': 'ANIMALS'}
--------------------------------------------------------------------------------
Score: 0.5
{'answer': 'Elephant', 'question': "It's the only living mammal in the order Proboseidea", 'category': 'ANIMALS'}
--------------------------------------------------------------------------------
Score: 0.4616832137107849
{'answer': 'the nose or snout', 'question': 'The gavial looks very much like a crocodile except for this bodily feature', 'category': 'ANIMALS'}
--------------------------------------------------------------------------------


## Comparison alphas' value

In [14]:
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    print(f"\nHybrid search alpha={alpha}")

    response = questions_hybrid.query.hybrid(
        query="animal",
        alpha=alpha,
        limit=3,
        return_properties=["question", "answer", "category"],
        return_metadata=MetadataQuery(score=True),
    )

    for obj in response.objects:
        print("Score:", obj.metadata.score)
        print(obj.properties)
        print("-" * 80)


Hybrid search alpha=0.0
Score: 1.0
{'answer': 'Antelope', 'question': 'Weighing around a ton, the eland is the largest species of this animal in Africa', 'category': 'ANIMALS'}
--------------------------------------------------------------------------------

Hybrid search alpha=0.25
Score: 0.9594089984893799
{'answer': 'Antelope', 'question': 'Weighing around a ton, the eland is the largest species of this animal in Africa', 'category': 'ANIMALS'}
--------------------------------------------------------------------------------
Score: 0.25
{'answer': 'Elephant', 'question': "It's the only living mammal in the order Proboseidea", 'category': 'ANIMALS'}
--------------------------------------------------------------------------------
Score: 0.23084160685539246
{'answer': 'the nose or snout', 'question': 'The gavial looks very much like a crocodile except for this bodily feature', 'category': 'ANIMALS'}
--------------------------------------------------------------------------------

Hybri

In [15]:
response = questions_hybrid.query.hybrid(
    query="space planet moon",
    alpha=0.5,
    limit=5,
    return_properties=["question", "answer", "category"],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print("Question:", obj.properties["question"])
    print("Answer:", obj.properties["answer"])
    print("Category:", obj.properties["category"])
    print("-" * 80)

Score: 0.5
Question: Changes in the tropospheric layer of this are what gives us weather
Answer: the atmosphere
Category: SCIENCE
--------------------------------------------------------------------------------
Score: 0.4278106391429901
Question: In 70-degree air, a plane traveling at about 1,130 feet per second breaks it
Answer: Sound barrier
Category: SCIENCE
--------------------------------------------------------------------------------
Score: 0.345834344625473
Question: It's the only living mammal in the order Proboseidea
Answer: Elephant
Category: ANIMALS
--------------------------------------------------------------------------------
Score: 0.34250500798225403
Question: This organ removes excess glucose from the blood & stores it as glycogen
Answer: Liver
Category: SCIENCE
--------------------------------------------------------------------------------
Score: 0.19016464054584503
Question: Weighing around a ton, the eland is the largest species of this animal in Africa
Answer: An

In [16]:
COLLECTION_NAME = "HardwareArticle"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

hardware_articles = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small"
    ),
    properties=[
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="category",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="component",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: HardwareArticle


In [17]:
hardware_data = [
    {
        "title": "CPU and Core Count",
        "content": "A CPU is the central processing unit of a computer. More cores can improve multitasking, compilation, virtualization, and parallel workloads.",
        "category": "Computer Hardware",
        "component": "CPU",
    },
    {
        "title": "GPU for Machine Learning",
        "content": "A GPU is useful for machine learning, deep learning, image processing, and parallel numeric computations. VRAM is important for large models.",
        "category": "Computer Hardware",
        "component": "GPU",
    },
    {
        "title": "RAM and System Performance",
        "content": "RAM stores temporary data used by running programs. More memory helps with large datasets, virtual machines, Docker containers, and development tools.",
        "category": "Computer Hardware",
        "component": "RAM",
    },
    {
        "title": "NVMe SSD Storage",
        "content": "An NVMe SSD provides fast storage for operating systems, applications, datasets, machine learning models, and project files.",
        "category": "Computer Hardware",
        "component": "Storage",
    },
    {
        "title": "Motherboard Expansion",
        "content": "A motherboard connects all computer components. PCIe slots, chipset features, RAM slots, and storage connectors define upgrade possibilities.",
        "category": "Computer Hardware",
        "component": "Motherboard",
    },
    {
        "title": "Power Supply Unit",
        "content": "A power supply unit provides stable power to the computer. GPU-heavy workstations require a reliable PSU with enough wattage.",
        "category": "Computer Hardware",
        "component": "PSU",
    },
]

result = hardware_articles.data.insert_many(hardware_data)

if result.errors:
    print("Errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [18]:
response = hardware_articles.query.fetch_objects(
    limit=10,
    return_properties=["title", "content", "category", "component"],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Title:", obj.properties["title"])
    print("Component:", obj.properties["component"])
    print("-" * 80)

UUID: 0b538918-70b9-40f7-838f-3961e36dec7a
Title: Motherboard Expansion
Component: Motherboard
--------------------------------------------------------------------------------
UUID: 398e9917-8781-4196-beba-e0a0e29b2dc2
Title: NVMe SSD Storage
Component: Storage
--------------------------------------------------------------------------------
UUID: 88b2de4d-299f-4286-9c0c-b7e298eac9c7
Title: CPU and Core Count
Component: CPU
--------------------------------------------------------------------------------
UUID: 8ad21ffd-901d-4e28-9652-0f1110fc5146
Title: RAM and System Performance
Component: RAM
--------------------------------------------------------------------------------
UUID: 9cc3b354-c6f6-4478-a454-255b68ac3c0f
Title: Power Supply Unit
Component: PSU
--------------------------------------------------------------------------------
UUID: ef7374bb-33ff-4843-9f6c-305c4bf2daed
Title: GPU for Machine Learning
Component: GPU
-----------------------------------------------------------------

## Vector search

In [19]:
response = hardware_articles.query.near_text(
    query="hardware for training machine learning models",
    limit=3,
    return_properties=["title", "content", "component"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Component:", obj.properties["component"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Distance: 0.4108467698097229
Title: GPU for Machine Learning
Component: GPU
Content: A GPU is useful for machine learning, deep learning, image processing, and parallel numeric computations. VRAM is important for large models.
--------------------------------------------------------------------------------
Distance: 0.5154964923858643
Title: NVMe SSD Storage
Component: Storage
Content: An NVMe SSD provides fast storage for operating systems, applications, datasets, machine learning models, and project files.
--------------------------------------------------------------------------------
Distance: 0.5655692219734192
Title: RAM and System Performance
Component: RAM
Content: RAM stores temporary data used by running programs. More memory helps with large datasets, virtual machines, Docker containers, and development tools.
--------------------------------------------------------------------------------


## BM25 keyword search

In [20]:
response = hardware_articles.query.bm25(
    query="GPU",
    query_properties=["title", "content", "component"],
    limit=3,
    return_properties=["title", "content", "component"],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print("Title:", obj.properties["title"])
    print("Component:", obj.properties["component"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Score: 2.0262882709503174
Title: GPU for Machine Learning
Component: GPU
Content: A GPU is useful for machine learning, deep learning, image processing, and parallel numeric computations. VRAM is important for large models.
--------------------------------------------------------------------------------
Score: 0.6929337382316589
Title: Power Supply Unit
Component: PSU
Content: A power supply unit provides stable power to the computer. GPU-heavy workstations require a reliable PSU with enough wattage.
--------------------------------------------------------------------------------


## Hybrid search

In [21]:
response = hardware_articles.query.hybrid(
    query="fast storage for datasets and machine learning projects",
    alpha=0.5,
    limit=3,
    return_properties=["title", "content", "component"],
    return_metadata=MetadataQuery(score=True, explain_score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print("Title:", obj.properties["title"])
    print("Component:", obj.properties["component"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Score: 1.0
Title: NVMe SSD Storage
Component: Storage
Content: An NVMe SSD provides fast storage for operating systems, applications, datasets, machine learning models, and project files.
--------------------------------------------------------------------------------
Score: 0.6093883514404297
Title: GPU for Machine Learning
Component: GPU
Content: A GPU is useful for machine learning, deep learning, image processing, and parallel numeric computations. VRAM is important for large models.
--------------------------------------------------------------------------------
Score: 0.3599211871623993
Title: RAM and System Performance
Component: RAM
Content: RAM stores temporary data used by running programs. More memory helps with large datasets, virtual machines, Docker containers, and development tools.
--------------------------------------------------------------------------------


In [22]:
query = "computer part important for AI models and parallel computations"

for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    print(f"\nHybrid search alpha={alpha}")

    response = hardware_articles.query.hybrid(
        query=query,
        alpha=alpha,
        limit=3,
        return_properties=["title", "content", "component"],
        return_metadata=MetadataQuery(score=True),
    )

    for obj in response.objects:
        print("Score:", obj.metadata.score)
        print("Title:", obj.properties["title"])
        print("Component:", obj.properties["component"])
        print("-" * 80)


Hybrid search alpha=0.0
Score: 1.0
Title: GPU for Machine Learning
Component: GPU
--------------------------------------------------------------------------------
Score: 0.2922108471393585
Title: CPU and Core Count
Component: CPU
--------------------------------------------------------------------------------
Score: 0.21804967522621155
Title: NVMe SSD Storage
Component: Storage
--------------------------------------------------------------------------------

Hybrid search alpha=0.25
Score: 1.0
Title: GPU for Machine Learning
Component: GPU
--------------------------------------------------------------------------------
Score: 0.3849269151687622
Title: CPU and Core Count
Component: CPU
--------------------------------------------------------------------------------
Score: 0.26078489422798157
Title: NVMe SSD Storage
Component: Storage
--------------------------------------------------------------------------------

Hybrid search alpha=0.5
Score: 1.0
Title: GPU for Machine Learning
Compo

In [23]:
client.close()